In [ ]:
# see link
# https://github.com/Fraud-Detection-Handbook

package ='08-sklearn.ensemble'
name='GradientBoosting'
tuningAndParameters='02-After tuning'

hyperparametersFound={ 
    'learning_rate': 0.01, 'max_depth': 6, 'n_estimators': 1673
}
scalerFound='None'


print('done')

In [ ]:
import sys
import os
from importlib import reload
fpath = os.path.join('..//scripts')
sys.path.append(fpath)

import warnings
warnings.filterwarnings('ignore')

#loading internal scripts
import datamanagement as dm
reload(dm)

import result as resultMd
reload(resultMd)

import graph as gf
reload(gf)

print('done')

In [ ]:
dfLearning, dfValidation =dm.getDataLearningAndValidation()

dfLearning.head()

In [ ]:
from sklearn.model_selection import train_test_split

TEST_SIZE = 0.20 # test size using_train_test_split
RANDOM_STATE = 0

predictors = dm.getPredictors(dfLearning)
target = dm.getTarget()

x_train, x_test, y_train, y_test = train_test_split(dfLearning[predictors], dfLearning[target], test_size = TEST_SIZE, 
                                                        stratify=dfLearning[target],
                                                        random_state = RANDOM_STATE)

In [ ]:
%%script false

from sklearn.ensemble import GradientBoostingClassifier
from scipy.stats import randint
from sklearn.model_selection import RandomizedSearchCV

dic_param={
    'n_estimators': randint(25,350),
    'max_depth': randint(2,12),
    'learning_rate':[0.01]
}
modelClf = GradientBoostingClassifier(random_state=42)

random_search = RandomizedSearchCV(modelClf,dic_param, scoring='f1', verbose=10,cv=4).fit(x_train, y_train)
print(random_search.best_params_)
print(random_search.best_score_)

#{'learning_rate': 0.01, 'max_depth': 6, 'n_estimators': 245}
#0.7551220148680369

In [ ]:
#%%script false

from sklearn.ensemble import GradientBoostingClassifier
from scipy.stats import randint
from sklearn.model_selection import GridSearchCV

dic_param={
    'n_estimators': [275,280,300,320,325],
    'max_depth': [8,9,10],
    'learning_rate':[0.05,0.1]
}
modelClf = GradientBoostingClassifier(random_state=42)

random_search = GridSearchCV(modelClf,dic_param, scoring='f1', verbose=10,cv=4).fit(x_train, y_train)
print(random_search.best_params_)
print(random_search.best_score_)

#{'learning_rate': 0.01, 'max_depth': 6, 'n_estimators': 245}
#0.7551220148680369

#{'learning_rate': 0.1, 'max_depth': 8, 'n_estimators': 245}
#0.8125784219792789

#{'learning_rate': 0.1, 'max_depth': 9, 'n_estimators': 300}
#0.830595347331496


In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import f1_score
from sklearn.ensemble import GradientBoostingClassifier
from datetime import datetime

modelClf = GradientBoostingClassifier(random_state=42)
parameters=hyperparametersFound

modelClf.set_params(**parameters)
then= datetime.now()
modelClf.fit(x_train, y_train)
now = datetime.now()
duration= now - then
duration_in_s = duration.total_seconds()
print("Duration ",duration_in_s)
resultMd.update_time_response_result(package, name, tuningAndParameters, duration_in_s)


predsTrain = modelClf.predict(x_train)
predsTest = modelClf.predict(x_test)

F1Learning =f1_score(y_train, predsTrain)
F1Test=f1_score(y_test, predsTest)
dm.show_confusion_matrix(y_train, predsTrain,'Confusion matrix learning data')
print(f"f1 train {F1Learning:.3f}")
dm.show_confusion_matrix(y_test, predsTest,'Confusion matrix test data')
print(f"f1 test {F1Test:.3f}")
resultMd.update_learning_test_result(package, name, tuningAndParameters, F1Learning,F1Test)

In [ ]:
gf.show_importance(modelClf, predictors)

In [ ]:
gf.show_prediction_graph(modelClf, x_test,y_test)

In [ ]:
predsValidation = modelClf.predict(dfValidation[predictors])
f1=f1_score(dfValidation[target], predsValidation)

dm.show_confusion_matrix(dfValidation[target], predsValidation,'Confusion matrix validation data')
print(f"f1 validation {f1:.3f}")
resultMd.update_performance_result(package, name, tuningAndParameters, f1)
resultMd.update_hyperparameters_result(package, name, tuningAndParameters, hyperparametersFound,scalerFound)